In [2]:
# https://twiki.cern.ch/twiki/bin/view/CMSPublic/RPCDPGResultsRPC2018#RPC_performance_with_Tracker_ext

import os, sys
sys.path.append("/users/hep/eigen1907/Workspace/Workspace-RPC/Modules")
from pathlib import Path
from NanoAODTnP.Plotting.PlotDetectorMap import plot_detector_map
from NanoAODTnP.Plotting.PlotEfficiency4Approval import *

store_path = Path('/users/hep/eigen1907/store/TnP-NanoAOD/analysis')
output_path = Path('/users/hep/eigen1907/Workspace/Workspace-RPC/Log/NanoAOD-TnP/240425-RPC2024/approval/proceeding')

In [3]:
input_path_2022 = store_path / 'without_blacklist' / 'Run2022.root'
input_path_2023 = store_path / 'without_blacklist' / 'Run2023.root'
hist_tnp_mass(input_path_2022, input_path_2023, output_path / 'tnp-mass.eps')

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


In [4]:
regions = ['Barrel', 'Endcap']
input_path_2022 = store_path / 'with_blacklist_roll' / 'Run2022.root'
input_path_2023 = store_path / 'with_blacklist_roll' / 'Run2023.root'   
for region in regions:
    hist_eff_by_roll(input_path_2022, input_path_2023, region, output_path / ('tnp-eff-' + region + '.eps'))

In [5]:
def get_eff(input_path, region):
    total_by_roll = uproot.open(f'{input_path}:total_by_roll').to_hist()
    passed_by_roll = uproot.open(f'{input_path}:passed_by_roll').to_hist()
    total = total_by_roll.values()
    passed = passed_by_roll.values()
    roll_name = np.array(total_by_roll.axes[0])
    region_params = get_region_params(region)
    is_iRPC = np.vectorize(lambda roll: roll in {"RE+4_R1_CH15_A", "RE+4_R1_CH16_A", "RE+3_R1_CH15_A", "RE+3_R1_CH16_A"})
    total = total[region_params['is_region'](roll_name) & ~is_iRPC(roll_name)]
    passed = passed[region_params['is_region'](roll_name) & ~is_iRPC(roll_name)]
    eff = np.divide(passed, total, out = np.zeros_like(total), where = (total > 0)) * 100
    eff = eff[total != 0]
    #n_total = len(total)
    #n_eff_under_70 = len(eff[eff <= 70])
    #n_excluded = len(total[total == 0])
    return eff

In [6]:
eff_2022 = get_eff(input_path_2022, 'All')
eff_2023 = get_eff(input_path_2023, 'All')

print(np.mean(eff_2022[eff_2022 > 70]))
print(np.mean(eff_2023[eff_2023 > 70]))

94.4680820725712
94.77989479787726


In [ ]:
def hist_probe_pt(input_path1, input_path2, output_path):
    probe_pt_2022 = uproot.open(f"{input_path1}:muon_tree/probe_pt").array(library="np")
    probe_pt_2023 = uproot.open(f"{input_path2}:muon_tree/probe_pt").array(library="np")

    mh.style.use(mh.styles.CMS)
    fig, ax = plt.subplots(figsize=(12, 8))
    #mh.cms.label(ax=ax, llabel=f'Preliminary', year='Run 3', com=13.6, loc=0, fontsize=24)
    mh.cms.label(ax=ax, data=True, year='Run 3', com=13.6, loc=0, fontsize=24)
    ax.set_xlabel(r'Probe Muon $p_T$ [$\mathit{GeV}$]', fontsize=22)
    ax.set_ylabel(r'Events / 0.5 $\mathit{GeV}$', fontsize=22)
    
    ax.set_xlim(0, 150)
    ax.ticklabel_format(style='sci', axis='y', scilimits=(0,0), useMathText=True)
    ax.yaxis.offsetText.set_visible(False)
    ax.annotate(r'$x10^{6}$', (-0.06, 1.0), #weight='bold',
                xycoords='axes fraction', fontsize=18, horizontalalignment='left')


    h_probe_pt_2022 = Hist(Regular(150, 0, 150))
    h_probe_pt_2023 = Hist(Regular(150, 0, 150))

    h_probe_pt_2022.fill(probe_pt_2022)
    h_probe_pt_2023.fill(probe_pt_2023)

    h_probe_pt_2022.plot(ax = ax,
                     yerr = False,
                     histtype="step",
                     edgecolor='firebrick',
                     linewidth=3,
                     hatch='\\\\',
                     flow=None,
                     label=r"$2022\ (34.7\ fb^{-1})$",
                     alpha=0.9)

    h_probe_pt_2023.plot(ax = ax,
                     yerr = False,
                     histtype="step",
                     edgecolor='mediumblue',
                     linewidth=3,
                     hatch='//',
                     flow=None,
                     label=r"$2023\ (27.9\ fb^{-1})$",
                     alpha=0.9)

    ax.legend(fontsize=24, handleheight = 1.2)

    if not output_path.parent.exists():
        output_path.parent.mkdir(parents=True)
    fig.savefig(output_path, format='png')
    plt.close(fig)

In [12]:
input_path_2022 = store_path / 'without_blacklist' / 'Run2022.root'
input_path_2023 = store_path / 'without_blacklist' / 'Run2023.root'
hist_probe_pt(input_path_2022, input_path_2023, output_path / 'probe_pt.png')